# Indexing and retrieval: look under the hood

Notebook 01 used a helper to build the index. Here you will build the same small corpus step by step and see how **chunk size**, **paper scope**, and **section filters** change the evidence a workflow can see. This is an optional deep dive after the guided tutorial.

In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb

papers = nb.classification_papers()

## 1. Split text into searchable chunks

The fixed-size chunker below uses **characters**, with overlap. It may cut a sentence in half. A paragraph chunker preserves this short section as one piece. We will index paragraph chunks so the retrieved passages stay readable.

In [ ]:
from epilens.rag.indexing.chunking import FixedSizeChunker, ParagraphChunker

text = papers["paper_methods"]["sections"][0].content
fixed_chunker = FixedSizeChunker(chunk_size=120, chunk_overlap=20)
paragraph_chunker = ParagraphChunker(min_chunk_size=40)
for name, pieces in [
    ("fixed size", fixed_chunker.chunk(text)),
    ("paragraph", paragraph_chunker.chunk(text)),
]:
    print(name, "→", len(pieces), "chunk(s)")
    for piece in pieces:
        print(" ", repr(piece))

## 2. Build a local index

An indexer turns sections into chunks, embeds each chunk, and stores the vector with its text and paper metadata. The tiny embedder counts a small vocabulary; it is deterministic and is only for teaching the API.

In [ ]:
from epilens.rag.indexing.indexer import Indexer
from epilens.rag.retrieval.candidates import SemanticCandidateRetriever
from epilens.rag.retrieval.retriever import Retriever
from epilens.vectordb.file import FileDB

embedder = nb.TinyKeywordEmbedder(nb.CLASSIFICATION_VOCABULARY)
work_dir = nb.bootstrap()  # Fresh index when this cell is rerun.
index = FileDB(str(work_dir / "index"))
indexer = Indexer(index, embedder=embedder, chunker=paragraph_chunker)
for paper_id, paper in papers.items():
    indexer.index_paper(paper["sections"], paper["metadata"], paper_id=paper_id)
index.save()

retriever = Retriever(
    index,
    candidate_retrievers=[SemanticCandidateRetriever(index, dense_embedder=embedder)],
    use_rerank=False,
)
print("Stored chunks:", len(index.get_points()))

Each stored point carries the text and fields such as `paper_id` and `section_type`. Those fields support both provenance and filtering.

In [ ]:
for point in index.get_points()[:3]:
    print(point["paper_id"], "|", point["section_type"], "|", point["text"][:100])

## 3. Search all papers, then one paper

A classifier searches **within the paper being classified**. Corpus search answers a different question: where in the whole collection might evidence appear?

In [ ]:
query = "reusable screening method tool"
for title, hits in [
    ("Corpus", retriever.retrieve(query, top_k=3)),
    ("Only paper_open_data", retriever.retrieve_by_paper(query, "paper_open_data", top_k=3)),
]:
    print(title)
    for hit in hits:
        print(f"  {hit.paper_id} | {hit.section_title}: {hit.text[:100]}")

## 4. Restrict the search to a section

Section filters are useful when a task asks about a specific statement, but they can exclude relevant evidence if a parser labels a section incorrectly.

In [ ]:
hits = retriever.retrieve(
    "where are the annotations or dataset available?",
    top_k=4,
    filter={"section_type": "Data availability"},
)
for hit in hits:
    print(f"{hit.paper_id} | {hit.section_title}: {hit.text}")

**Try it:** use `fixed_chunker` in the `Indexer`, rerun from the index cell, and compare the retrieved passages. In a real corpus, inspect samples before fixing chunking and retrieval settings for the whole run.